**K-Means: clustering de personalidades.**<br><br>
Ejemplo didáctico de K-Means en Python sobre perfiles de famosos (`data/analisis.csv`).<br>
La versión script del mismo pipeline está en `kMeans.py`. Documentación: `docs/k-means.md`.


In [ ]:
# Primero hacemos los imports necesarios.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# Declaramos las constantes (alineadas con kMeans.py)
CSV_FILE = "./data/analisis.csv"
FIRST_N_ROWS = 20
MAX_CLUSTERS_ELBOW = 20
DEFAULT_N_CLUSTERS = 5
RANDOM_STATE = 42

Primero tenemos que cargar  los datos...

In [ ]:
dataFrame = pd.read_csv(CSV_FILE)

Imprimimos el resumen de los datos cargados

In [ ]:
dataFrame.describe()

Imprimimos los N primeros registros...

In [ ]:
print(f"{dataFrame.head(FIRST_N_ROWS)}")

El fichero contiene 9 categorias: <br>
<ol> 
    <li>Actor/ Actriz</li>
    <li>Cantante</li>
    <li>Modelo</li>
    <li>Tv, series</li>
    <li>Radio</li>
    <li>Tecnologia</li>
    <li>Deportes</li>
    <li>Politica</li>
    <li>Escritor</li>
</ol><br>
Cada categoria tiene los siguientes registros:

In [ ]:
print(f"{dataFrame.groupby('categoria').size()}")

Vamos a ver como están distribuidos los datos en las categorias del dataFrame ( menos la categoria que es lo que hemos visto antes...)

In [ ]:
dataFrame.drop(['categoria'], axis = 'columns').hist(figsize=(10,8))

Vamos a hacer un plot Cruzado entre tres de sus carácterísticas a ver si encontramos algún tipo de relación entre las distintas columnas.

In [ ]:
sb.pairplot(
    dataFrame.dropna(),
    hue = 'categoria',
    height = 4,
    vars = ["op", "ex", "ag"],
    kind = "scatter"
    )

Parece que no hay correlación alguna entre los usuarios y sus categorias y eso lo vemos en que las formas de las curvas no son similares. Cada uno crece en un punto diferente al resto, es decir, ninguna curva incluye a otra curva con el mismo comportamiento. (a priori)<br>
Vamos a pintar los datos como puntos en una gráfica

In [ ]:
X = np.array(dataFrame[["op","ex","ag"]])
y = np.array(dataFrame['categoria'])

fig = plt.figure(figsize=(15,9))
# Plotamos en 3D
ax = Axes3D(fig)
# Un array de colores
colores= ['blue','red','green','blue','cyan','yellow','orange','black','pink','brown','purple']
asignar = []
# Para cada categoria, seleccionamos un color del "buffer" de colores.
for row in y:
    asignar.append(colores[row])

ax.scatter(X[:,0], X[:,1], X[:,2], c= asignar, s=60)
# He creado el eje pero no se lo he añadido a la figura.
fig.add_axes(ax)
plt.show()

Procedemos a encontrar el número óptimo de clusters bucando el "codo" el punto donde el score ya no aumenta significativamente.

In [ ]:
nc = range(1, MAX_CLUSTERS_ELBOW)
# Creamos los n clasificadores K means, uno por cada numero de cluster.
kmeans = [
  KMeans(n_clusters=i, n_init=10, random_state=RANDOM_STATE) for i in nc
]
# Calculo el Score para cada clasificador creado.
score = [kmeans[i].fit(X).score(X) for i in range(len(kmeans))]
# Pasamos a plotar los scores
plt.plot(nc, score)
plt.xlabel("Numero de Clusters")
plt.ylabel("Score")
plt.title("Elbow Curve")
plt.show()

Viendo la gráfica... Podemos decir que a partir de 5 clusters, el score no aumenta significativamente por lo que solo aumentaríamos el coste computacional sin ganar mucho score. Nos quedamos con 5 clusters.

Ejecutamos el clasificador K-Means con 5 clusters

In [ ]:
kmeans = KMeans(
  n_clusters=DEFAULT_N_CLUSTERS,
  n_init=10,
  random_state=RANDOM_STATE,
).fit(X)
# Obtenemos los centroides
centroids = kmeans.cluster_centers_
print(f"Centroides: {centroids}", end="\n\n")
labels = kmeans.labels_
print(f"Labels: {labels}", end="\n\n")

Graficamos el entrenamiento segun la clase asignada.

In [ ]:
labels = kmeans.predict(X)
C = kmeans.cluster_centers_
colores = ['red', 'green', 'blue', 'cyan', 'yellow']
asignar = []
for row in labels:
    asignar.append(colores[row])
fig = plt.figure()
ax = Axes3D(fig)
# Plotamos los puntos con el color SEGUN la clasificacion
ax.scatter(X[:,0], X[:,1], X[:,2], c = asignar, s= 60)
ax.scatter(C[:,0], C[:,1], C[:,2], marker = '*', c= colores, s= 1000)
fig.add_axes(ax)
plt.show()

Vemos los centroides como estrellas y los puntos que le "pertenecen" en su mismo color.

Ahora plotamos los centroides y sus puntos en función de cada par de columnas de las 3 que hemos elegido.

In [ ]:
f1 = dataFrame['op'].values
f2 = dataFrame['ex'].values
# Plotamos los puntos de op respecto ex
plt.scatter(f1,f2, c = asignar, s= 70)
# Plotamos los centroides
plt.scatter(C[:,0], C[:,1], marker = '*', c = colores, s= 1000)
plt.show()

In [ ]:
f1 = dataFrame['op'].values
f2 = dataFrame['ag'].values
# Plotamos los puntos de op respecto ex
plt.scatter(f1,f2, c = asignar, s= 70)
# Plotamos los centroides
plt.scatter(C[:,0], C[:,2], marker = '*', c = colores, s= 1000)
plt.show()

In [ ]:
f1 = dataFrame['ex'].values
f2 = dataFrame['ag'].values
# Plotamos los puntos de op respecto ex
plt.scatter(f1,f2, c = asignar, s= 70)
# Plotamos los centroides
plt.scatter(C[:,1], C[:,2], marker = '*', c = colores, s= 1000)
plt.show()

Vemos que los centroides representan bastante bien a sus puntos.